In [ ]:
import pandas as pd
df = pd.read_csv('/content/cleaned_training_with_missing_indicators_imputed.csv')
df.head()

,Test_ID,Applied_Voltage_kV,Load_Current_A,Ambient_Temperature_C,Test_Duration_min,Sensor_S1,Sensor_S2,Sensor_S3,Sensor_S4,Reference_Parameter,Validity_Label,Sensor_S1_missing,Sensor_S2_missing,Sensor_S3_missing,Sensor_S4_missing
0,TRN-0889,16.7196,93.1228,33.3421,19.9546,13.6343,15.1361,16.5062,62.9115,34.8501,Valid,0,0,0,0
1,TRN-0820,21.4477,82.5230,34.4915,47.0553,15.6466,15.9849,19.3289,39.8667,30.4762,Valid,0,0,0,0
2,TRN-0411,19.2432,107.3619,29.4673,5.2763,15.1080,16.6481,18.3834,44.5390,46.8046,Valid,0,0,0,0
3,TRN-0754,22.7191,69.9690,26.9695,20.3051,14.9296,14.7026,18.6269,44.2673,23.7153,Valid,0,0,0,0
4,TRN-0707,13.5008,108.8999,35.5415,29.1939,12.6845,15.2455,14.6132,44.0720,48.5994,Valid,0,0,0,0


In [ ]:
import numpy as np
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier

num_cols = ['Applied_Voltage_kV','Load_Current_A','Ambient_Temperature_C',
            'Test_Duration_min','Sensor_S1','Sensor_S2','Sensor_S3','Sensor_S4']

X = df[num_cols]
y_reg = df['Reference_Parameter']
y_clf = (df['Validity_Label'] == 'Valid').astype(int)

In [ ]:
reg_baseline = -cross_val_score(RandomForestRegressor(random_state=42), X, y_reg,
                                 cv=5, scoring='neg_mean_absolute_error').mean()

clf_baseline = cross_val_score(RandomForestClassifier(random_state=42), X, y_clf,
                                cv=5, scoring='f1').mean()

print("Baseline MAE (Reference_Parameter):", reg_baseline)
print("Baseline F1 (Validity):", clf_baseline)

Baseline MAE (Reference_Parameter): 1.0512698686785722
Baseline F1 (Validity): 0.9638817220297543


In [ ]:
def evaluate_features(feature_cols, label):
    X_new = df[feature_cols]
    mae = -cross_val_score(RandomForestRegressor(random_state=42), X_new, y_reg,
                            cv=5, scoring='neg_mean_absolute_error').mean()
    f1 = cross_val_score(RandomForestClassifier(random_state=42), X_new, y_clf,
                          cv=5, scoring='f1').mean()
    print(f"[{label}] MAE: {mae:.4f} (baseline {reg_baseline:.4f}) | F1: {f1:.4f} (baseline {clf_baseline:.4f})")
    return mae, f1

In [ ]:
df['V_times_I'] = df['Applied_Voltage_kV'] * df['Load_Current_A']
evaluate_features(num_cols + ['V_times_I'], "V_times_I")

[V_times_I] MAE: 1.0319 (baseline 1.0513) | F1: 0.9682 (baseline 0.9639)


(np.float64(1.0318967361333338), np.float64(0.968156550636107))

In [ ]:
df['Current_sq'] = df['Load_Current_A'] ** 2
evaluate_features(num_cols + ['Current_sq'], "Current_sq")

[Current_sq] MAE: 1.0484 (baseline 1.0513) | F1: 0.9671 (baseline 0.9639)


(np.float64(1.0483893696404762), np.float64(0.9670826834728544))

In [ ]:
df['Voltage_sq'] = df['Applied_Voltage_kV'] ** 2
evaluate_features(num_cols + ['Voltage_sq'], "Voltage_sq")

[Voltage_sq] MAE: 1.0498 (baseline 1.0513) | F1: 0.9682 (baseline 0.9639)


(np.float64(1.0498195417214289), np.float64(0.9681655247349363))

In [ ]:
df['Current_x_Duration'] = df['Load_Current_A'] * df['Test_Duration_min']
evaluate_features(num_cols + ['Current_x_Duration'], "Current_x_Duration")

[Current_x_Duration] MAE: 1.0433 (baseline 1.0513) | F1: 0.9671 (baseline 0.9639)


(np.float64(1.0433280492000003), np.float64(0.9670856498880616))

In [ ]:
df['S1_minus_S2'] = df['Sensor_S1'] - df['Sensor_S2']
df['S1_minus_S3'] = df['Sensor_S1'] - df['Sensor_S3']
df['S2_minus_S3'] = df['Sensor_S2'] - df['Sensor_S3']
evaluate_features(num_cols + ['S1_minus_S2','S1_minus_S3','S2_minus_S3'], "Sensor diffs")

[Sensor diffs] MAE: 1.0685 (baseline 1.0513) | F1: 0.9903 (baseline 0.9639)


(np.float64(1.0684876848880953), np.float64(0.9902707897744468))

In [ ]:
df['Sensor_avg'] = df[['Sensor_S1','Sensor_S2','Sensor_S3','Sensor_S4']].mean(axis=1)
df['Sensor_max'] = df[['Sensor_S1','Sensor_S2','Sensor_S3','Sensor_S4']].max(axis=1)
df['Sensor_min'] = df[['Sensor_S1','Sensor_S2','Sensor_S3','Sensor_S4']].min(axis=1)
evaluate_features(num_cols + ['Sensor_avg','Sensor_max','Sensor_min'], "Sensor avg/max/min")

[Sensor avg/max/min] MAE: 1.0680 (baseline 1.0513) | F1: 0.9687 (baseline 0.9639)


(np.float64(1.0680056676023817), np.float64(0.968678361615218))

In [ ]:
def evaluate_regression(feature_cols, label):
    mae = -cross_val_score(RandomForestRegressor(random_state=42), df[feature_cols], y_reg,
                            cv=5, scoring='neg_mean_absolute_error').mean()
    print(f"[{label}] MAE: {mae:.4f} (baseline {reg_baseline:.4f})")
    return mae

def evaluate_classification(feature_cols, label):
    f1 = cross_val_score(RandomForestClassifier(random_state=42), df[feature_cols], y_clf,
                          cv=5, scoring='f1').mean()
    print(f"[{label}] F1: {f1:.4f} (baseline {clf_baseline:.4f})")
    return f1

In [ ]:
reg_features = num_cols + ['V_times_I', 'Current_sq', 'Voltage_sq', 'Current_x_Duration']
evaluate_regression(reg_features, "Combined regression features")

[Combined regression features] MAE: 1.0287 (baseline 1.0513)


np.float64(1.0286876821000004)

In [ ]:
clf_features = num_cols + ['S1_minus_S2', 'S1_minus_S3', 'S2_minus_S3']
evaluate_classification(clf_features, "Combined classification features")

[Combined classification features] F1: 0.9903 (baseline 0.9639)


np.float64(0.9902707897744468)

## Re-testing with Gradient Boosting (Person 2 & 3's actual model)

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import GradientBoostingRegressor, GradientBoostingClassifier

df = pd.read_csv('/content/cleaned_training_with_missing_indicators_imputed.csv')
df.head()

,Test_ID,Applied_Voltage_kV,Load_Current_A,Ambient_Temperature_C,Test_Duration_min,Sensor_S1,Sensor_S2,Sensor_S3,Sensor_S4,Reference_Parameter,Validity_Label,Sensor_S1_missing,Sensor_S2_missing,Sensor_S3_missing,Sensor_S4_missing
0,TRN-0889,16.7196,93.1228,33.3421,19.9546,13.6343,15.1361,16.5062,62.9115,34.8501,Valid,0,0,0,0
1,TRN-0820,21.4477,82.5230,34.4915,47.0553,15.6466,15.9849,19.3289,39.8667,30.4762,Valid,0,0,0,0
2,TRN-0411,19.2432,107.3619,29.4673,5.2763,15.1080,16.6481,18.3834,44.5390,46.8046,Valid,0,0,0,0
3,TRN-0754,22.7191,69.9690,26.9695,20.3051,14.9296,14.7026,18.6269,44.2673,23.7153,Valid,0,0,0,0
4,TRN-0707,13.5008,108.8999,35.5415,29.1939,12.6845,15.2455,14.6132,44.0720,48.5994,Valid,0,0,0,0


In [ ]:
num_cols = ['Applied_Voltage_kV','Load_Current_A','Ambient_Temperature_C',
            'Test_Duration_min','Sensor_S1','Sensor_S2','Sensor_S3','Sensor_S4']

X = df[num_cols]
y_reg = df['Reference_Parameter']
y_clf = (df['Validity_Label'] == 'Valid').astype(int)

In [ ]:
reg_baseline = -cross_val_score(GradientBoostingRegressor(random_state=42), X, y_reg,
                                 cv=5, scoring='neg_mean_absolute_error').mean()
clf_baseline = cross_val_score(GradientBoostingClassifier(random_state=42), X, y_clf,
                                cv=5, scoring='f1').mean()

print("Baseline MAE (Reference_Parameter):", reg_baseline)
print("Baseline F1 (Validity):", clf_baseline)

Baseline MAE (Reference_Parameter): 0.9640314644155014
Baseline F1 (Validity): 0.9632239372632767


In [ ]:
def evaluate_features(feature_cols, label):
    X_new = df[feature_cols]
    mae = -cross_val_score(GradientBoostingRegressor(random_state=42), X_new, y_reg,
                            cv=5, scoring='neg_mean_absolute_error').mean()
    f1 = cross_val_score(GradientBoostingClassifier(random_state=42), X_new, y_clf,
                          cv=5, scoring='f1').mean()
    print(f"[{label}] MAE: {mae:.4f} (baseline {reg_baseline:.4f}) | F1: {f1:.4f} (baseline {clf_baseline:.4f})")
    return mae, f1

In [ ]:
df['V_times_I'] = df['Applied_Voltage_kV'] * df['Load_Current_A']
df['Current_sq'] = df['Load_Current_A'] ** 2
df['Voltage_sq'] = df['Applied_Voltage_kV'] ** 2
df['Current_x_Duration'] = df['Load_Current_A'] * df['Test_Duration_min']
df['S1_minus_S2'] = df['Sensor_S1'] - df['Sensor_S2']
df['S1_minus_S3'] = df['Sensor_S1'] - df['Sensor_S3']
df['S2_minus_S3'] = df['Sensor_S2'] - df['Sensor_S3']
df['Sensor_avg'] = df[['Sensor_S1','Sensor_S2','Sensor_S3','Sensor_S4']].mean(axis=1)
df['Sensor_max'] = df[['Sensor_S1','Sensor_S2','Sensor_S3','Sensor_S4']].max(axis=1)
df['Sensor_min'] = df[['Sensor_S1','Sensor_S2','Sensor_S3','Sensor_S4']].min(axis=1)

In [ ]:
evaluate_features(num_cols + ['V_times_I'], "V_times_I (GB)")
evaluate_features(num_cols + ['Current_sq'], "Current_sq (GB)")
evaluate_features(num_cols + ['Voltage_sq'], "Voltage_sq (GB)")
evaluate_features(num_cols + ['Current_x_Duration'], "Current_x_Duration (GB)")
evaluate_features(num_cols + ['S1_minus_S2','S1_minus_S3','S2_minus_S3'], "Sensor diffs (GB)")
evaluate_features(num_cols + ['Sensor_avg','Sensor_max','Sensor_min'], "Sensor avg/max/min (GB)")

[V_times_I (GB)] MAE: 0.9291 (baseline 0.9640) | F1: 0.9605 (baseline 0.9632)
[Current_sq (GB)] MAE: 0.9653 (baseline 0.9640) | F1: 0.9627 (baseline 0.9632)
[Voltage_sq (GB)] MAE: 0.9642 (baseline 0.9640) | F1: 0.9627 (baseline 0.9632)
[Current_x_Duration (GB)] MAE: 0.9649 (baseline 0.9640) | F1: 0.9610 (baseline 0.9632)
[Sensor diffs (GB)] MAE: 0.9738 (baseline 0.9640) | F1: 0.9818 (baseline 0.9632)
[Sensor avg/max/min (GB)] MAE: 0.9629 (baseline 0.9640) | F1: 0.9658 (baseline 0.9632)


(np.float64(0.9629319052291055), np.float64(0.9657859377984865))

In [ ]:
reg_features = num_cols + ['V_times_I', 'Current_sq', 'Voltage_sq', 'Current_x_Duration']
evaluate_features(reg_features, "Combined regression features (GB)")

clf_features = num_cols + ['S1_minus_S2', 'S1_minus_S3', 'S2_minus_S3']
evaluate_features(clf_features, "Combined classification features (GB)")

[Combined regression features (GB)] MAE: 0.9384 (baseline 0.9640) | F1: 0.9643 (baseline 0.9632)
[Combined classification features (GB)] MAE: 0.9738 (baseline 0.9640) | F1: 0.9818 (baseline 0.9632)


(np.float64(0.9737603730438071), np.float64(0.9817590778188903))

In [ ]:
from sklearn.model_selection import cross_val_score

# Baseline accuracy
acc_baseline = cross_val_score(GradientBoostingClassifier(random_state=42), X, y_clf,
                                cv=5, scoring='accuracy').mean()
print(f"Baseline Accuracy: {acc_baseline*100:.2f}%")

Baseline Accuracy: 93.40%


In [ ]:
def check_accuracy(feature_cols, label):
    X_new = df[feature_cols]
    acc = cross_val_score(GradientBoostingClassifier(random_state=42), X_new, y_clf,
                           cv=5, scoring='accuracy').mean()
    print(f"[{label}] Accuracy: {acc*100:.2f}% (baseline {acc_baseline*100:.2f}%)")
    return acc

In [ ]:
check_accuracy(num_cols + ['S1_minus_S2','S1_minus_S3','S2_minus_S3'], "Sensor diffs")
check_accuracy(num_cols + ['Sensor_avg','Sensor_max','Sensor_min'], "Sensor avg/max/min")
check_accuracy(num_cols + ['V_times_I'], "V_times_I")

[Sensor diffs] Accuracy: 96.80% (baseline 93.40%)
[Sensor avg/max/min] Accuracy: 93.90% (baseline 93.40%)
[V_times_I] Accuracy: 92.90% (baseline 93.40%)


np.float64(0.929)

In [ ]:
for seed in [0, 1, 7]:
    acc = cross_val_score(GradientBoostingClassifier(random_state=seed),
                           df[clf_features], y_clf, cv=5, scoring='f1').mean()
    print(f"Seed {seed}: F1 = {acc:.4f}")

Seed 0: F1 = 0.9823
Seed 1: F1 = 0.9829
Seed 7: F1 = 0.9829


In [ ]:
check_accuracy(num_cols + ['Sensor_S1_missing','Sensor_S2_missing','Sensor_S3_missing','Sensor_S4_missing'], "Missing indicators")

[Missing indicators] Accuracy: 93.90% (baseline 93.40%)


np.float64(0.9389999999999998)

In [ ]:
# Save the training data with your engineered columns included
export_cols = ['Test_ID'] + num_cols + ['Reference_Parameter', 'Validity_Label',
               'V_times_I', 'Current_sq', 'Voltage_sq', 'Current_x_Duration',
               'S1_minus_S2', 'S1_minus_S3', 'S2_minus_S3']

df[export_cols].to_csv('training_data_with_engineered_features.csv', index=False)

from google.colab import files
files.download('training_data_with_engineered_features.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# CSV for Person 2 — Reference Parameter (regression) features only
person2_cols = ['Test_ID'] + num_cols + ['Reference_Parameter',
                'V_times_I', 'Current_sq', 'Voltage_sq', 'Current_x_Duration']

df[person2_cols].to_csv('training_data_for_person2_reference.csv', index=False)

# CSV for Person 3 — Validity Label (classification) features only
person3_cols = ['Test_ID'] + num_cols + ['Validity_Label',
                'S1_minus_S2', 'S1_minus_S3', 'S2_minus_S3']

df[person3_cols].to_csv('training_data_for_person3_validity.csv', index=False)

from google.colab import files
files.download('training_data_for_person2_reference.csv')
files.download('training_data_for_person3_validity.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>